In [1]:
import numpy as np
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.cosmology import FlatLambdaCDM
from astroquery.sdss import SDSS
from scipy.ndimage import gaussian_filter, label

In [62]:
query = """
SELECT TOP 1000000
    p.ra, p.dec, s.z
FROM SpecObj AS s
JOIN PhotoObj AS p ON s.bestobjiD = p.objid
WHERE s.class = 'GALAXY'
    AND s.z > 0.01 AND s.z < 0.1
    AND s.zWarning = 0
"""

print("before query sent")
result = SDSS.query_sql(query)
print("after query sent")


before query sent


InconsistentTableError: Number of header columns (1) inconsistent with data columns in data line 13

In [58]:
result.colnames

['ra', 'dec', 'z']

In [59]:
ra = result['ra']
dec = result['dec']
z = result['z']

sky_coords =SkyCoord(ra=ra * u.deg, dec = dec*u.deg, frame = "icrs")

In [ ]:
cosmo = FlatLambdaCDM(H0=67.66 * u.km / u.s / u.Mpc, Om0=0.31)

In [27]:
comoving_distance = cosmo.comoving_distance(z)

In [28]:
x_coord = comoving_distance * np.cos(sky_coords.dec.rad) * np.cos(sky_coords.ra.rad)
y_coord = comoving_distance * np.cos(sky_coords.dec.rad) * np.sin(sky_coords.ra.rad)
z_coord = comoving_distance * np.sin(sky_coords.dec.rad)

In [29]:
galaxy_positions = np.column_stack((x_coord,y_coord,z_coord))

In [30]:
counts, edges = np.histogramdd(galaxy_positions, bins = 200)

smoothed_density = gaussian_filter(counts, sigma=0.5)

mean_density = np.mean(smoothed_density)
void_threshold = 0.2 * mean_density
underdense_mask = smoothed_density < void_threshold
voids, num_voids = label(underdense_mask, structure = np.ones((3,3,3)))

num_voids

143